# Exploratory Data Analysis: Tech Addiction Prediction
Welcome to this EDA notebook! The goal of this analysis is to thoroughly understand the dataset provided for predicting tech addiction (`addicted_label`). 

**Why do we do EDA?**
1. **Understand the Data:** Get a feel for distributions, missing values, and potential outliers.
2. **Feature Interactions:** Discover how different variables relate to each other and to the target variable.
3. **Inform Feature Engineering:** Find clues that will help us create better features to improve model performance.
4. **Model Selection:** Decide on preprocessing steps and select appropriate algorithms based on data characteristics.

Let's get started by importing our tools.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up beautiful and consistent visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14


## 1. Data Loading and Initial Overview
First, we load our data and take a quick look at its shape and the first few rows. This helps us confirm the data loaded correctly and gives us an initial glimpse into the features we are working with.


In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e8/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e8/test.csv')
sample_sub = pd.read_csv('/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv')

print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape: {test_df.shape}\n")

display(train_df.head())


Let's look at the data types and basic statistics. 
**What to look for:**
- Are numerical columns correctly identified as such (int/float)?
- Are there categorical columns stored as objects?
- Are the scales of different numerical features vastly different? (If so, scaling might be needed for certain models).


In [ ]:
train_df.info()


In [ ]:
train_df.describe().T


## 2. Missing Values Analysis
Missing data is a common issue in real-world datasets. 
**Why analyze this?**
Depending on the percentage and pattern of missing values, we might choose to:
- **Drop rows/columns:** If a huge portion is missing.
- **Impute:** Fill missing values with mean, median, mode, or using an predictive model.
- **Leave as-is:** Tree-based models (like XGBoost, LightGBM) can handle missing values natively, which can sometimes be the best approach!


In [ ]:
# Calculate missing percentages
missing_percent = (train_df.isnull().sum() / len(train_df)) * 100
missing_percent = missing_percent[missing_percent > 0].sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=missing_percent.index, y=missing_percent.values, palette='viridis')
plt.title('Percentage of Missing Values per Feature')
plt.ylabel('Missing Percentage (%)')
plt.xlabel('Features')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(missing_percent)


## 3. Target Variable Analysis
Our target variable is `addicted_label`. Let's check its distribution.
**Why is this critical?**
If the classes are highly imbalanced (e.g., 90% non-addicted, 10% addicted), we would need to:
- Use stratified splitting for validation.
- Choose robust evaluation metrics (like F1-score, ROC-AUC) instead of plain Accuracy.
- Potentially use techniques like class weighting, SMOTE, or under-sampling.


In [ ]:
plt.figure(figsize=(6, 5))
ax = sns.countplot(data=train_df, x='addicted_label', palette='Set2')
plt.title('Distribution of Target Variable (addicted_label)')
plt.ylabel('Count')
plt.xlabel('Addicted Label')

# Add percentages on top of bars
total = len(train_df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2 - 0.05
    y = p.get_height() + (p.get_height() * 0.02)
    ax.annotate(percentage, (x, y), ha='center')

plt.show()


## 4. Univariate Analysis
Let's explore individual features. We will start with the numerical features to understand their distributions.
**What to look for:**
- **Skewness:** Are the distributions heavily skewed? Models like Linear Regression or Neural Networks often benefit from log transformations on skewed data.
- **Outliers:** Are there extreme values that might throw off our models?


In [ ]:
# List of key numerical features
num_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 
            'gaming_hours', 'work_study_hours', 'sleep_hours', 
            'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time']

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(train_df[col].dropna(), kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()


Now, let's look at the categorical features.


In [ ]:
cat_cols = ['gender', 'stress_level', 'academic_work_impact']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(cat_cols):
    sns.countplot(data=train_df, x=col, ax=axes[i], palette='pastel')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()


## 5. Bivariate Analysis: Features vs Target
This is where we hunt for signals! We want to see how each feature behaves differently for people who are addicted (1) versus those who are not (0).

**Why do this?**
Features that show a clear separation between the target classes will be very powerful for our model. If a feature looks identical for both classes, it might not be very useful.


In [ ]:
# Boxplots for numerical features vs Target
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=train_df, x='addicted_label', y=col, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} by Addiction Label')

plt.tight_layout()
plt.show()


Let's also visualize categorical features against the target using stacked bar charts or grouped count plots.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(cat_cols):
    # Calculate proportions
    prop_df = train_df.groupby(col)['addicted_label'].value_counts(normalize=True).rename('proportion').reset_index()
    sns.barplot(data=prop_df, x=col, y='proportion', hue='addicted_label', ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} vs Addiction Proportion')

plt.tight_layout()
plt.show()


## 6. Correlation Matrix
Finally, let's check for linear correlations between numerical variables.

**What to look for:**
- **Feature vs Target:** High correlation (positive or negative) with `addicted_label` indicates a strong predictor.
- **Multicollinearity (Feature vs Feature):** If two features are highly correlated with each other (e.g., `daily_screen_time_hours` and `app_opens_per_day`), they might be redundant. Some models (like Linear/Logistic Regression) struggle with multicollinearity, while Tree models are generally fine.


In [ ]:
plt.figure(figsize=(12, 10))

# Compute correlation matrix, dropping non-numerical columns
corr_matrix = train_df[num_cols + ['addicted_label']].corr()

# Create a mask to hide the upper triangle for cleaner visualization
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=.5, cbar_kws={"shrink": .75})
plt.title('Correlation Matrix of Numerical Features')
plt.show()


## 7. Conclusions & Feature Engineering Ideas
Based on the analysis above, here are some actionable insights for our next steps:

### Feature Engineering Clues:
1. **Total Activity Time:** Can we create a new feature that sums up `social_media_hours`, `gaming_hours`, and `work_study_hours`? Does this total equal `daily_screen_time_hours`? If not, the difference could be a meaningful new feature (e.g., `other_screen_time`).
2. **Weekend vs Weekday:** A feature like `weekend_vs_weekday_ratio = weekend_screen_time / daily_screen_time_hours` could indicate if someone binges on weekends.
3. **App Engagement Intensity:** `app_opens_per_hour = app_opens_per_day / daily_screen_time_hours`. A high number might indicate a shorter attention span or compulsive checking behavior.
4. **Encoding Categoricals:** `stress_level` has a natural order (Low -> Medium -> High). We should map this ordinally (0, 1, 2) rather than using One-Hot Encoding, to preserve the relationship. `gender` and `academic_work_impact` can be One-Hot Encoded.

### Model Selection Clues:
- **Missing Values:** We have several columns with missing values. We can either build an imputation pipeline (e.g., SimpleImputer, KNNImputer) or choose a gradient boosting model (XGBoost, LightGBM, CatBoost) which handles NaNs organically.
- **Non-Linear Relationships:** Given human behavior data, relationships are likely non-linear. Tree-based ensembles are highly recommended as our primary modeling choice.
